In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        _tok = ""
        try:
            from google.colab import userdata
            _tok = userdata.get("GH_TOKEN") or ""
        except Exception:
            _tok = ""
        if not _tok:
            print("WARNING: no 'GH_TOKEN' Colab secret found; cloning this PRIVATE repo will fail.\n"
                  "Add a GitHub token (repo scope) via the key icon (Secrets) as 'GH_TOKEN', then re-run.")
        _url = (f"https://{_tok}@github.com/{_slug}.git" if _tok
                else f"https://github.com/{_slug}.git")
        subprocess.run(["git", "clone", "--depth", "1", _url, str(_root)], check=True)
        subprocess.run(["git", "-C", str(_root), "remote", "set-url", "origin",
                        f"https://github.com/{_slug}.git"])  # keep the token out of the saved remote
    os.chdir(_root / "07-application-agent-framework/long-running-durable/lra-core/lra-core/notebooks/solutions")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 02 · Waiting for days, resuming from anywhere

Human-in-the-loop is the defining feature of long-running agents. The run sleeps for free and is woken by an event.

In [1]:
import sys; sys.path[:0] = ["..", "../.."]      # repo root, from notebooks/ or notebooks/solutions/
from core import Engine, Queue, Store, Crash, drain
from workflow import STEPS, CALLS
clock = [0.0]; now = lambda: clock[0]              # a clock we control: days pass in one line
def fresh(steps=STEPS):
    CALLS.clear(); return Engine(Store(), Queue(), dict(steps), clock=now, lease_ttl=60)


## 1. Suspend

The `review` step returns a *wait* outcome. Check what the run looks like while it sleeps: no task, no lease, nothing to bill.

In [2]:
engine = fresh()
engine.start("run-1", "draft", {"topic": "agents"})
print(drain(engine, engine.queue, now))
run = engine.store.get("run-1")
print(run["status"], run["wait"])
assert run["status"] == "WAITING" and engine.queue.tasks == [] and run["lease"] is None

[('draft', 'ok'), ('review', 'waiting')]
WAITING {'key': 'approval:run-1', 'then': 'publish', 'timeout': 259200.0}


## 2. Resume — two days later, from a different machine

The approval arrives as a webhook: `POST /runs/run-1/events {key, payload}`. Fill in the key it must carry.

In [3]:
clock[0] += 2 * 86400
resumed = engine.resume("run-1", "approval:run-1", {"decision": "approve", "by": "editor"})
print("resumed at step:", resumed["step"], "| queued:", engine.queue.tasks)
print(drain(engine, engine.queue, now))
assert engine.store.get("run-1")["result"]["published"] is True

resumed at step: publish | queued: [(172800.0, 'run-1', 'publish', 1)]
[('publish', 'ok'), ('notify', 'succeeded')]


## 3. Duplicates and wrong keys are not errors

Webhooks get sent twice; links get clicked twice. What must `resume` return for a second delivery?

In [4]:
engine = fresh()
engine.start("run-2", "draft", {"topic": "x"}); drain(engine, engine.queue, now)
print(engine.resume("run-2", "approval:someone-else", {"decision": "approve"}))
first = engine.resume("run-2", "approval:run-2", {"decision": "approve"})
second = engine.resume("run-2", "approval:run-2", {"decision": "approve"})
assert first is not None and second is None and len(engine.queue.tasks) == 1

None


## 4. Timeouts are absolute

The wait stores `timeout = now + 3 days` *at suspension time*, so it survives restarts. The reaper (Cloud Scheduler → `/reap`) enforces it.

In [5]:
engine = fresh()
engine.start("run-3", "draft", {"topic": "x"}); drain(engine, engine.queue, now)
print("day 1:", engine.reap())
clock[0] += 3 * 86400 + 1
print("later:", engine.reap())
assert engine.store.get("run-3")["status"] == "FAILED"

day 1: []
later: [('timeout', 'run-3')]


## 5. Your turn: a rejection path

Write a `publish_or_not` step that reads the decision from `ctx.state["events"][key]` and finishes with `{"published": False}` on reject — no retry, no effect.

In [6]:
def ask(ctx):
    return ("wait", f"ok:{ctx.run_id}", "publish_or_not")
def publish_or_not(ctx):
    decision = ctx.state["events"][f"ok:{ctx.run_id}"]
    if decision["decision"] != "approve":
        return ("done", {"published": False})
    ctx.effect("publish", lambda: CALLS.append(("publish", "x")))
    return ("done", {"published": True})

engine = fresh({"ask": ask, "publish_or_not": publish_or_not})
engine.start("run-4", "ask", {}); drain(engine, engine.queue, now)
engine.resume("run-4", "ok:run-4", {"decision": "reject"}); drain(engine, engine.queue, now)
assert engine.store.get("run-4")["result"] == {"published": False} and CALLS == []
print("rejected cleanly:", engine.store.get("run-4")["history"])

rejected cleanly: [('ask', 1, 'ok', 'wait'), ('publish_or_not', 1, 'ok', 'done')]
